# 🩺 From Population Models to Personalised Digital Twins
### Summer School on AI for Diabetes Management · UdG · July 2026

---

## What you will do

| # | Section | Your role |
|---|---------|----------|
| 1 | Setup & dependencies | ▶ Run |
| 2 | Load patient data from Excel | ▶ Run |
| 3 | See the problem | ▶ Run |
| 4 | Load the pre-trained population model | ▶ Run |
| 5 | cVAE architecture | ✏️ Fill 4 blanks |
| 6 | Fine-tuning loop | ✏️ Fill 3 blanks |
| 7 | Evaluate & compare | ▶ Run |
| 8 | Digital twin: what-if simulation | ▶ Run + explore |

**Always run cells top to bottom.** If your session restarts, re-run from Section 1.

---

## The one-sentence framing

> A model pretrained on 58 patients captures general glucose–therapy dynamics.  
> Fine-tuning on **one patient's data** personalises it into their digital twin — in under a minute.

This notebook walks through that pipeline using the architecture from:  
> *Mujahid et al. — "Personalised Digital Twins for T1D Using Generative AI" (2025)*

---
## 📦 Section 1 — Setup
*Run these cells. No blanks.*

In [ ]:
!pip install tensorflow==2.15.0 scipy openpyxl -q
print('✅ Dependencies ready')

In [ ]:
# ─── Download workshop files from GitHub ─────────────────────────────────────
# Run this once at the start of every session.
# Everything downloads in ~10 seconds.

import os

BASE = 'https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main'

# Pretrained decoder
if not os.path.exists('vae_decoder_epoch_006.h5'):
    !wget -q {BASE}/vae_decoder_epoch_006.h5
    print('✅ Decoder downloaded')
else:
    print('✅ Decoder already present')

# Patient Excel files
os.makedirs('patients', exist_ok=True)
for fname in ['patient_1.xlsx', 'patient_2.xlsx']:   # add more as needed
    fpath = f'patients/{fname}'
    if not os.path.exists(fpath):
        !wget -q -O {fpath} {BASE}/patients/{fname}
        print(f'✅ {fname} downloaded')
    else:
        print(f'✅ {fname} already present')

print('\n🎯 All files ready — proceed to Section 2')

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras import Model, Input, backend as K
from tensorflow.keras.layers import (
    Dense, LSTM, Concatenate, Reshape, Flatten, Dropout,
    TimeDistributed, LeakyReLU, LayerNormalization
)
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.initializers import RandomNormal

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
np.random.seed(42);  tf.random.set_seed(42)
print(f'✅ TensorFlow {tf.__version__}')

---
## 📂 Section 2 — Load Patient Data

Each file in the `patients/` folder is one patient's CGM data in Excel format.

**Expected columns:** `ID`, `BG` (blood glucose, mg/dL), `PI` (plasma insulin effect), `RA` (carbohydrate absorption rate)

**Normalization** follows `vae_personalize.py`: each patient's signals are independently scaled to [0, 1] using their own min/max. This is *patient-specific* normalization — important because we must undo it later to interpret predictions in mg/dL.

**Sequence construction** follows `data_processing.py`:
- Input at time *t*: `(PI_t, RA_t)` — scalar therapy values
- Target: `BG_{t+1 .. t+18}` — 90 minutes of future glucose (18 × 5-min steps)

*Run. No blanks.*

In [ ]:
# ─── Configuration — edit these paths if needed ───────────────────────────────
PATIENTS_DIR   = 'patients'          # folder containing patient_1.xlsx, patient_2.xlsx ...
DECODER_PATH   = 'vae_decoder_epoch_006.h5'   # pretrained population decoder

HORIZON        = 18    # prediction horizon (18 × 5 min = 90 min)
LATENT_DIM     = 5     # must match the pretrained decoder
HIDDEN         = 64    # must match the pretrained decoder
HOLDOUT_DAYS   = 3     # days reserved for evaluation (not seen during fine-tuning)
SAMPLES_PER_DAY = 288  # 5-min CGM resolution

print(f'Patients folder : {PATIENTS_DIR}')
print(f'Decoder file    : {DECODER_PATH}')
print(f'Horizon         : {HORIZON} steps ({HORIZON*5} min)')
print(f'Latent dim      : {LATENT_DIM}')

In [ ]:
# ─── Data loading & preprocessing ────────────────────────────────────────────

def load_patient_excel(filepath):
    """
    Load one patient Excel file and return a cleaned DataFrame.
    Expected columns: ID, BG, PI, RA
    """
    df = pd.read_excel(filepath)
    required = {'BG', 'PI', 'RA'}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f'{filepath}: missing columns {missing}')
    return df[['BG', 'PI', 'RA']].dropna().reset_index(drop=True)


def preprocess_patient(df):
    """
    Apply per-patient MinMaxScaler to BG, PI, RA.
    Returns normalized DataFrame + scalers (needed to invert BG later).

    Follows vae_personalize.py:
        scaler_bg  = MinMaxScaler().fit_transform(df[['BG']])
        scaler_ins = MinMaxScaler().fit_transform(df[['PI']])
        scaler_car = MinMaxScaler().fit_transform(df[['RA']])
    """
    df = df.copy()
    scaler_bg  = MinMaxScaler()
    scaler_ins = MinMaxScaler()
    scaler_car = MinMaxScaler()

    df['BG'] = scaler_bg .fit_transform(df[['BG']]).astype('float32')
    df['PI'] = scaler_ins.fit_transform(df[['PI']]).astype('float32')
    df['RA'] = scaler_car.fit_transform(df[['RA']]).astype('float32')

    return df, scaler_bg, scaler_ins, scaler_car


def make_sequences(df, horizon=HORIZON):
    """
    Build (insulin, carbs, bg_future) sequence pairs.
    Mirrors DataProcessor.create_and_save_pairs().
    Vectorised with numpy for speed on large files.

    Returns:
        insulin : (N, 1, 1)  float32
        carbs   : (N, 1, 1)  float32
        bg      : (N, 1, 18) float32
    """
    pi  = df['PI'].values.astype('float32')
    ra  = df['RA'].values.astype('float32')
    bg  = df['BG'].values.astype('float32')
    N   = len(bg) - horizon

    ins_arr  = pi[:N].reshape(N, 1, 1)
    carb_arr = ra[:N].reshape(N, 1, 1)
    # Build BG windows using stride tricks (no Python loop)
    idx      = np.arange(horizon)[None, :] + np.arange(1, N+1)[:, None]  # (N, 18)
    bg_arr   = bg[idx].reshape(N, 1, horizon)

    return ins_arr, carb_arr, bg_arr


# ─── Discover and load all patient files ──────────────────────────────────────
patient_files = sorted([
    f for f in os.listdir(PATIENTS_DIR)
    if f.startswith('patient_') and f.endswith('.xlsx')
])

if not patient_files:
    raise FileNotFoundError(f'No patient_*.xlsx files found in {PATIENTS_DIR}/')

print(f'Found {len(patient_files)} patient file(s): {patient_files}\n')

patients = {}   # key: patient label, value: dict with data

for fname in patient_files:
    label = fname.replace('.xlsx', '')   # e.g. 'patient_1'
    fpath = os.path.join(PATIENTS_DIR, fname)

    df_raw          = load_patient_excel(fpath)
    df_norm, sc_bg, sc_ins, sc_car = preprocess_patient(df_raw)
    ins, carb, bg   = make_sequences(df_norm)

    # Split: all but last HOLDOUT_DAYS for training, last HOLDOUT_DAYS for evaluation
    holdout_n = HOLDOUT_DAYS * SAMPLES_PER_DAY
    if len(bg) <= holdout_n:
        print(f'[WARN] {label}: not enough data for holdout split — skipping')
        continue

    tr_ins,  tr_carb,  tr_bg  = ins[:-holdout_n],  carb[:-holdout_n],  bg[:-holdout_n]
    ho_ins,  ho_carb,  ho_bg  = ins[-holdout_n:],  carb[-holdout_n:],  bg[-holdout_n:]

    patients[label] = dict(
        df_raw=df_raw, df_norm=df_norm,
        scaler_bg=sc_bg,
        tr_ins=tr_ins, tr_carb=tr_carb, tr_bg=tr_bg,
        ho_ins=ho_ins, ho_carb=ho_carb, ho_bg=ho_bg,
        n_train_days = len(tr_bg) // SAMPLES_PER_DAY,
        n_total_days = len(bg)    // SAMPLES_PER_DAY,
    )

    print(f'{label}: {len(df_raw):,} rows  →  '
          f'train={tr_bg.shape[0]:,} ({len(tr_bg)//SAMPLES_PER_DAY}d)  '
          f'holdout={ho_bg.shape[0]:,} ({HOLDOUT_DAYS}d)')

print(f'\n✅ {len(patients)} patient(s) loaded')


In [ ]:
# ─── Pick which patient to personalise ───────────────────────────────────────
# If there are multiple files, change PATIENT_ID to the one you want to work with.

PATIENT_ID = list(patients.keys())[0]   # default: first patient
P = patients[PATIENT_ID]

print(f'Working with: {PATIENT_ID}')
print(f'  Total data  : {P["n_total_days"]} days')
print(f'  Training    : {P["n_train_days"]} days')
print(f'  Holdout     : {HOLDOUT_DAYS} days (evaluation only)')

---
## 👀 Section 3 — See the Problem
*Run. No blanks.*

In [ ]:
# ─── Visualise the patient's data ────────────────────────────────────────────
df_raw  = P['df_raw']
df_norm = P['df_norm']
sc_bg   = P['scaler_bg']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# A — Raw CGM trace (first 3 days)
ax = axes[0]
show = min(3 * SAMPLES_PER_DAY, len(df_raw))
ax.plot(df_raw['BG'].values[:show], color='steelblue', lw=1.2)
ax.axhspan(70, 180, alpha=0.12, color='green', label='Target range (70–180 mg/dL)')
ax.set_title(f'A — CGM trace: {PATIENT_ID} (first 3 days)', fontweight='bold')
ax.set_xlabel('Time step (5 min)'); ax.set_ylabel('BG (mg/dL)')
ax.legend(fontsize=8)

# B — BG distribution
ax = axes[1]
ax.hist(df_raw['BG'].values, bins=50, color='steelblue', alpha=0.8, density=True)
ax.axvspan(70, 180, alpha=0.12, color='green')
ax.set_title('B — BG distribution (full record)', fontweight='bold')
ax.set_xlabel('BG (mg/dL)'); ax.set_ylabel('Density')

# C — Insulin and carb absorption over 1 day
ax = axes[2]
show1 = min(SAMPLES_PER_DAY, len(df_raw))
t = np.arange(show1)
ax2 = ax.twinx()
ax .plot(t, df_raw['PI'].values[:show1], color='darkorange', lw=1.2, label='Insulin (PI)')
ax2.plot(t, df_raw['RA'].values[:show1], color='purple',     lw=1.2, label='Carbs (RA)', alpha=0.7)
ax .set_title('C — Therapy inputs (1 day)', fontweight='bold')
ax .set_xlabel('Time step (5 min)')
ax .set_ylabel('Insulin effect',  color='darkorange')
ax2.set_ylabel('Carb absorption', color='purple')
ax .legend(loc='upper left',  fontsize=8)
ax2.legend(loc='upper right', fontsize=8)

# Summary stats
bg = df_raw['BG'].values
tir = ((bg >= 70) & (bg <= 180)).mean() * 100
tir_str  = f'Time in range (70–180): {tir:.1f}%'
mean_str = f'Mean BG: {bg.mean():.1f} mg/dL  |  SD: {bg.std():.1f} mg/dL'

plt.suptitle(f'{PATIENT_ID}   ·   {mean_str}   ·   {tir_str}',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

print('\nQuestion: Does this patient look like a typical population average?')
print('That tension — individual vs population — is what this workshop is about.')

---
## 🎁 Section 4 — Load the Pre-Trained Population Model

Training on 58 patients under a leave-one-patient-out protocol takes hours.  
We have done that for you. The file `vae_decoder_epoch_006.h5` is the best checkpoint  
from early stopping — the decoder that best captures population-level glucose dynamics.

**What we load and why:**
- We load the **decoder only** — the encoder is always reinitialised for each new patient (see Section 6).
- We extract the weights so we can transfer them into a fresh decoder instance that we control.

*Run. No blanks.*

In [ ]:
# ─── Load pretrained population decoder ──────────────────────────────────────
# This mirrors vae_personalize.py → load_and_transfer_decoder_weights()

if not os.path.exists(DECODER_PATH):
    raise FileNotFoundError(
        f'Could not find {DECODER_PATH}. '
        f'Make sure the file is in the same folder as this notebook.'
    )

pop_dec_loaded = load_model(DECODER_PATH, compile=False)

# Store the weights — we will inject them into our own decoder in Section 6
PRETRAINED_DECODER_WEIGHTS = pop_dec_loaded.get_weights()

print(f'✅ Loaded: {DECODER_PATH}')
print(f'   Decoder has {len(PRETRAINED_DECODER_WEIGHTS)} weight tensors')
print(f'   Input  shapes: {[w.shape for w in PRETRAINED_DECODER_WEIGHTS[:3]]}')
print()
print('These weights encode population-level glucose–therapy dynamics.')
print('They are the starting point for every patient-specific fine-tuning.')

---
## 🏗️ Section 5 — Build the cVAE Architecture

### Architecture overview

```
ENCODER                                        DECODER
──────────────────────────────────────────     ────────────────────────────────────────────────
BG_{t+1..t+18} ─┐                             z  ──► Dense ──► baseline trajectory
Insulin_t       ─┼─► project ──► LSTM ──► z                                          ► + ──► output
Carbs_t         ─┘              (μ, σ²)        (Ins, Carb) ──► softplus gains
                                               (sign-constrained: ins↓ glucose, carbs↑ glucose)
```

**Why a cVAE and not a plain LSTM?**  
An LSTM predicts *one* future trajectory. The cVAE models a *distribution* over trajectories — sampling different `z` values gives different plausible futures. This is essential for uncertainty quantification and for the what-if simulations in Section 8.

**Monotonicity constraint (key clinical design decision):**  
The decoder's control branch uses `softplus` to enforce that the insulin gain is always negative and the carb gain is always positive. This prevents the model from ever predicting that more insulin raises glucose — a biologically impossible (and clinically dangerous) prediction.

---

### ✏️ 4 blanks

| Blank | Concept tested |
|-------|----------------|
| 1 | Which layer summarises a sequence into a single vector? |
| 2 | Reparameterisation trick: z = μ + σ·ε |
| 3 | Monotonicity constraint on insulin gain |
| 4 | KL divergence formula |


In [ ]:
# ─── Encoder ──────────────────────────────────────────────────────────────────
# Inputs:
#   in_ins  : insulin at time t          shape (batch, 1, 1)
#   in_carb : carbs at time t            shape (batch, 1, 1)
#   in_bg   : future BG t+1..t+18        shape (batch, 1, 18)
#
# Outputs:
#   z_mean    : (batch, LATENT_DIM)
#   z_log_var : (batch, LATENT_DIM)
#
# The three inputs are each projected to width HIDDEN, stacked into a short
# sequence of length 3, then summarised by a recurrent layer.

def build_encoder(latent_dim=LATENT_DIM, hidden=HIDDEN):
    in_ins  = Input(shape=(1, 1),        name='enc_ins')
    in_carb = Input(shape=(1, 1),        name='enc_carb')
    in_bg   = Input(shape=(1, HORIZON),  name='enc_bg')

    # Project each input to the same width, reshape to (batch, 1, hidden)
    ins_proj  = Reshape((1, hidden))(Dense(hidden)(Flatten()(in_ins)))
    carb_proj = Reshape((1, hidden))(Dense(hidden)(Flatten()(in_carb)))
    bg_proj   = Reshape((1, hidden))(Dense(hidden)(Flatten()(in_bg)))

    # Stack into a sequence: (batch, 3, hidden)
    seq = Concatenate(axis=1)([ins_proj, carb_proj, bg_proj])

    # ✏️ BLANK 1 ──────────────────────────────────────────────────────────────
    # seq has shape (batch, 3, hidden) — a sequence of length 3.
    # We need a single summary vector (not a sequence) for the latent projection.
    # Which Keras layer is built to process a sequence and return one vector?
    # Hint: it also appears in the GAN critic in the paper.
    # Set return_sequences=False so it outputs (batch, hidden), not (batch, 3, hidden).
    h = ???(hidden, return_sequences=False)(seq)
    # ─────────────────────────────────────────────────────────────────────────

    z_mean    = Dense(latent_dim, name='z_mean')(h)
    z_log_var = Dense(latent_dim, name='z_log_var')(h)

    return Model([in_ins, in_carb, in_bg], [z_mean, z_log_var], name='encoder')


# ─── Reparameterisation trick ─────────────────────────────────────────────────
# The VAE samples z ~ q(z|x) during training.
# To backpropagate through sampling, we write:
#   z = μ + σ · ε,   ε ~ N(0, I)
# where σ = exp(0.5 · log_var),  since log_var = log(σ²)

def sampling(args):
    z_mean, z_log_var = args
    eps = tf.random.normal(shape=tf.shape(z_mean))

    # ✏️ BLANK 2 ──────────────────────────────────────────────────────────────
    # Return z = μ + σ · ε
    # σ = exp(0.5 · z_log_var)
    # Hint: tf.exp(...)
    return z_mean + ??? * eps
    # ─────────────────────────────────────────────────────────────────────────

print('✅ Encoder defined — run next cell for decoder')

In [ ]:
# ─── Decoder ──────────────────────────────────────────────────────────────────
# Inputs:
#   in_z    : latent sample z             shape (batch, LATENT_DIM)
#   in_ins  : insulin at time t           shape (batch, 1, 1)
#   in_carb : carbs at time t             shape (batch, 1, 1)
#
# Output: predicted BG trajectory         shape (batch, 1, 18)
#
# Two-branch design (Figure 1, paper):
#   Branch 1 (baseline) : z → Dense → trajectory
#   Branch 2 (control)  : (ins, carb) → sign-constrained gains
#   Final output = baseline + ins_scalar * ins_gain + carb_scalar * carb_gain
#
# This mirrors gan_pretrain_model.py:
#   w_ins  = -(K.softplus(w_ins_raw)  + self.min_gain_mgdl)   # always negative
#   w_carb =  (K.softplus(w_carb_raw) + self.min_gain_mgdl)   # always positive

MIN_GAIN = 0.01   # minimum gain magnitude — prevents dead gains

def build_decoder(latent_dim=LATENT_DIM, horizon=HORIZON, hidden=HIDDEN):
    in_z    = Input(shape=(latent_dim,), name='dec_z')
    in_ins  = Input(shape=(1, 1),        name='dec_ins')
    in_carb = Input(shape=(1, 1),        name='dec_carb')

    # ── Branch 1: baseline from latent vector ────────────────────────────────
    h        = Dense(hidden, activation='relu')(in_z)
    baseline = Reshape((1, horizon))(Dense(horizon)(h))    # (batch, 1, 18)

    # ── Branch 2: therapy control signal ─────────────────────────────────────
    ins_flat  = Flatten()(in_ins)     # (batch, 1)
    carb_flat = Flatten()(in_carb)    # (batch, 1)

    ins_raw  = Dense(horizon)(ins_flat)    # (batch, 18)  unconstrained
    carb_raw = Dense(horizon)(carb_flat)   # (batch, 18)  unconstrained

    # ✏️ BLANK 3 ──────────────────────────────────────────────────────────────
    # Apply the monotonicity constraint.
    # carb_gain is done for you as a reference — it must be POSITIVE.
    # ins_gain must be NEGATIVE (insulin lowers BG).
    #
    # softplus(x) = log(1 + exp(x)) → always positive
    # To get a negative gain: negate softplus.
    #
    # Complete ins_gain using the same pattern as carb_gain but negated.
    carb_gain = Reshape((1,horizon))( tf.math.softplus(carb_raw) + MIN_GAIN )  # positive ✓
    ins_gain  = Reshape((1,horizon))( ???                                    )  # ← your turn
    # ─────────────────────────────────────────────────────────────────────────

    ins_scalar  = Reshape((1,1))(ins_flat)
    carb_scalar = Reshape((1,1))(carb_flat)
    control = ins_scalar * ins_gain + carb_scalar * carb_gain

    output = baseline + control    # (batch, 1, 18)

    return Model([in_z, in_ins, in_carb], output, name='decoder')


# ─── VAE loss ─────────────────────────────────────────────────────────────────
# L = reconstruction + β · KL
#
# Reconstruction: MSE between real and predicted BG horizon
# KL divergence : KL[ q(z|x) ‖ N(0,I) ]
#               = −½ · Σ( 1 + log_var − mean² − exp(log_var) )
#
# β (kl_weight) = 3×10⁻³  — from vae_train.py

def vae_loss(bg_true, bg_pred, z_mean, z_log_var, kl_weight=3e-3):
    recon = tf.reduce_mean(tf.square(bg_true - bg_pred))

    # ✏️ BLANK 4 ──────────────────────────────────────────────────────────────
    # KL divergence: KL = −½ · mean( 1 + z_log_var − z_mean² − exp(z_log_var) )
    # Fill in the three ??? placeholders.
    kl = -0.5 * tf.reduce_mean(
        1.0 + ??? - tf.square(???) - tf.exp(???)
    )
    # ─────────────────────────────────────────────────────────────────────────

    return recon + kl_weight * kl, recon, kl

print('✅ Decoder + loss defined — run next cell to verify')

In [ ]:
# ─── Verify architecture (run after filling all 4 blanks) ────────────────────

encoder = build_encoder()
decoder = build_decoder()

# Shape check
dummy_ins  = np.zeros((4,1,1),  dtype='float32')
dummy_carb = np.zeros((4,1,1),  dtype='float32')
dummy_bg   = np.zeros((4,1,18), dtype='float32')

zm, zlv = encoder([dummy_ins, dummy_carb, dummy_bg])
z       = sampling([zm, zlv])
out     = decoder([z, dummy_ins, dummy_carb])

print(f'Encoder  →  z_mean: {zm.shape}  z_log_var: {zlv.shape}')
print(f'z sample →  {z.shape}')
print(f'Decoder  →  {out.shape}  (expected (4, 1, {HORIZON}))')

# Monotonicity check: more insulin → lower predicted BG
z0   = np.zeros((1, LATENT_DIM), dtype='float32')
c0   = np.zeros((1, 1, 1),      dtype='float32')
p_lo = decoder([z0, np.zeros((1,1,1),'float32'), c0]).numpy().mean()
p_hi = decoder([z0, np.ones((1,1,1), 'float32'), c0]).numpy().mean()

ok = p_hi < p_lo
print(f'\nMonotonicity (↑ insulin → ↓ BG): {"✅ PASS" if ok else "❌ FAIL — re-check Blank 3"}')
print(f'  pred(ins=0) = {p_lo:.4f},  pred(ins=1) = {p_hi:.4f}')

# Transfer pretrained weights into our decoder
try:
    decoder.set_weights(PRETRAINED_DECODER_WEIGHTS)
    print('\n✅ Pretrained population weights loaded into decoder')
except Exception as e:
    print(f'\n❌ Weight transfer failed: {e}')
    print('   The decoder architecture must exactly match the pretrained model.')
    print(f'   Check LATENT_DIM={LATENT_DIM}, HIDDEN={HIDDEN}, HORIZON={HORIZON}')

---
## 🎯 Section 6 — Fine-Tuning: Personalise for the Patient

### Strategy (from `vae_personalize.py`)

```
Step 1  Fresh encoder          — no population weights; learns patient-specific latent space
Step 2  Decoder ← population   — warm-started from pretrained weights (the prior)
Step 3  Fine-tune both         — on the patient's training data only
```

The encoder is **always reinitialised** for each new patient.  
The decoder is **warm-started** — it brings population knowledge, needing far fewer updates.

### ✏️ 3 blanks

In [ ]:
# ─── Training helpers ─────────────────────────────────────────────────────────

def _train_step(ins, carb, bg, enc, dec, opt, kl_w=3e-3):
    with tf.GradientTape() as tape:
        zm, zlv    = enc([ins, carb, bg], training=True)
        z          = sampling([zm, zlv])
        bg_pred    = dec([z, ins, carb],  training=True)
        loss, _, _ = vae_loss(bg, bg_pred, zm, zlv, kl_w)
    grads = tape.gradient(loss, enc.trainable_variables + dec.trainable_variables)
    opt.apply_gradients(zip(grads, enc.trainable_variables + dec.trainable_variables))
    return loss


def run_training(ins, carb, bg, enc, dec,
                 n_epochs=10, batch_size=32, lr=5e-4, tag=''):
    """
    Mini training loop — mirrors vae_personalize.py → vae_model.pretrain().
    Uses the same hyperparameters: batch_size=32, early_stopping patience=3.
    """
    opt = Adam(learning_rate=lr)
    N   = ins.shape[0]
    history = []
    for ep in range(n_epochs):
        idx    = np.random.permutation(N)
        losses = []
        for s in range(0, N, batch_size):
            bi = idx[s:s+batch_size]
            if len(bi) < 2: continue
            l = _train_step(
                tf.constant(ins[bi]), tf.constant(carb[bi]),
                tf.constant(bg[bi]),  enc, dec, opt
            )
            losses.append(float(l))
        mean_l = float(np.mean(losses))
        history.append(mean_l)
        print(f'  [{tag}] epoch {ep+1}/{n_epochs}  loss={mean_l:.5f}')
    return history


def eval_wasserstein(ins, carb, bg, enc, dec, n_samples=10):
    """Wasserstein-1 distance between real and generated BG distributions."""
    preds = []
    for _ in range(n_samples):
        zm, zlv = enc([ins, carb, bg], training=False)
        z       = sampling([zm, zlv])
        preds.append(dec([z, ins, carb], training=False).numpy())
    pred_mean = np.mean(preds, axis=0)
    return wasserstein_distance(bg.reshape(-1), pred_mean.reshape(-1)), pred_mean

print('✅ Training helpers defined')


In [ ]:
# ─── Fine-tuning ──────────────────────────────────────────────────────────────

tr_ins  = P['tr_ins']
tr_carb = P['tr_carb']
tr_bg   = P['tr_bg']


# ── Step 1: Build the fine-tuning models ──────────────────────────────────────

# ✏️ BLANK 5 ──────────────────────────────────────────────────────────────────
# Build a FRESH encoder for this patient.
# Do NOT reuse 'encoder' from Section 5 — that one was used only for verification.
# A fresh encoder starts with random weights and will learn a
# patient-specific latent representation during fine-tuning.
ft_enc = ???()
# ─────────────────────────────────────────────────────────────────────────────

# ✏️ BLANK 6 ──────────────────────────────────────────────────────────────────
# Build a new decoder and load the pretrained population weights into it.
# Two lines:
#   Line 1: build_decoder()                          — fresh architecture
#   Line 2: .set_weights(PRETRAINED_DECODER_WEIGHTS) — inject population prior
# This mirrors vae_personalize.py → load_and_transfer_decoder_weights()
ft_dec = ???()
ft_dec.???(PRETRAINED_DECODER_WEIGHTS)
# ─────────────────────────────────────────────────────────────────────────────

print(f'Fine-tuning encoder : fresh weights ✓')
print(f'Fine-tuning decoder : population prior loaded ✓')
print(f'Training on {len(tr_bg):,} samples ({P["n_train_days"]} days) from {PATIENT_ID}\n')


# ── Step 2: Measure population model performance before fine-tuning ───────────
# The population model = pretrained decoder + z=0 (the prior mean).
# This is the prediction before any patient-specific adaptation.
def eval_population_baseline(ins, carb, bg, dec, n_samples=10):
    """Evaluate pretrained decoder using z sampled from N(0,I) — the prior mean."""
    preds = []
    for _ in range(n_samples):
        z = tf.random.normal((len(ins), LATENT_DIM))
        preds.append(dec([z, ins, carb], training=False).numpy())
    pred_mean = np.mean(preds, axis=0)
    return wasserstein_distance(bg.reshape(-1), pred_mean.reshape(-1)), pred_mean

w_pop, pred_pop = eval_population_baseline(tr_ins, tr_carb, tr_bg, ft_dec)
print(f'Population model  →  Wasserstein = {w_pop:.5f}  (no personalisation yet)\n')


# ── Step 3: Fine-tune ─────────────────────────────────────────────────────────
print('⏳ Fine-tuning...')
t0 = time.perf_counter()

ft_history = run_training(
    tr_ins, tr_carb, tr_bg,
    ft_enc, ft_dec,
    n_epochs=10, batch_size=32, lr=5e-4,
    tag=f'Fine-tune {PATIENT_ID}'
)

elapsed = time.perf_counter() - t0
print(f'\n✅ Fine-tuning complete in {elapsed:.1f}s  ({elapsed/60:.2f} min)')
print('   Paper reports sub-minute personalisation on real data. How does yours compare?')


# ── Step 4: Measure personalised model performance ────────────────────────────
w_ft, pred_ft = eval_wasserstein(tr_ins, tr_carb, tr_bg, ft_enc, ft_dec)
print(f'\nPopulation model  →  Wasserstein = {w_pop:.5f}')
print(f'Fine-tuned model  →  Wasserstein = {w_ft:.5f}')
print(f'Improvement       :  {(w_pop - w_ft) / w_pop * 100:.1f}%')


In [ ]:
# ─── No-pretraining baseline ──────────────────────────────────────────────────
# This model trains from scratch on the patient's data only — no population prior.
# It represents the counterfactual: what if we had never pretrained on the population?
# Expected result: worse than fine-tuned, demonstrating the value of transfer learning.

# ✏️ BLANK 7 ──────────────────────────────────────────────────────────────────
# Build TWO fresh models — encoder AND decoder — with no pretrained weights.
# Train on the same patient data with the same hyperparameters as fine-tuning.
sc_enc = ???()
sc_dec = ???()

print(f'⏳ Training from scratch (no pretraining)...')
sc_history = run_training(
    ???, ???, ???,
    sc_enc, sc_dec,
    n_epochs=???, batch_size=???, lr=???,
    tag='Scratch'
)
# ─────────────────────────────────────────────────────────────────────────────

w_sc, pred_sc = eval_wasserstein(tr_ins, tr_carb, tr_bg, sc_enc, sc_dec)
print(f'\nScratch model  →  Wasserstein = {w_sc:.5f}')

---
## 📊 Section 7 — Evaluate on Holdout Data
*Run. No blanks.*

In [ ]:
# ─── Holdout evaluation ───────────────────────────────────────────────────────
# The holdout set is the last HOLDOUT_DAYS of the patient's record.
# None of the models have seen this data.

ho_ins  = P['ho_ins']
ho_carb = P['ho_carb']
ho_bg   = P['ho_bg']

w_pop_ho, pred_pop_ho = eval_population_baseline(ho_ins, ho_carb, ho_bg, ft_dec)
w_ft_ho,  pred_ft_ho  = eval_wasserstein(ho_ins, ho_carb, ho_bg, ft_enc,       ft_dec)
w_sc_ho,  pred_sc_ho  = eval_wasserstein(ho_ins, ho_carb, ho_bg, sc_enc,       sc_dec)

print('\n' + '='*54)
print(f"  {'Model':<30} {'Wasserstein ↓':>12}")
print('='*54)
print(f"  {'🔵 Population (no fine-tuning)':<30} {w_pop_ho:>12.5f}")
print(f"  {'🟠 Fine-tuned (pretrained prior)':<30} {w_ft_ho:>12.5f}")
print(f"  {'🟢 Scratch (no pretraining)':<30} {w_sc_ho:>12.5f}")
print('='*54)

best = min(w_pop_ho, w_ft_ho, w_sc_ho)
winner = {w_pop_ho: 'Population', w_ft_ho: 'Fine-tuned', w_sc_ho: 'Scratch'}[best]
print(f'\n🏆 Best on holdout: {winner}')
if winner == 'Fine-tuned':
    print('   Transfer learning pays off — the population prior helped personalisation.')
elif winner == 'Scratch':
    print('   Scratch wins here. The population prior may need more epochs, or this')
    print('   patient is well-behaved enough that the prior adds little noise.')
else:
    print('   Population model wins. This patient is well-represented by the population.')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# A — BG distributions on holdout
ax = axes[0,0]
real_flat = ho_bg.reshape(-1)
ax.hist(real_flat,              bins=50, density=True, alpha=0.8, color='black',      label='Real patient')
ax.hist(pred_pop_ho.reshape(-1),bins=50, density=True, alpha=0.4, color='steelblue',  label=f'Population  W={w_pop_ho:.4f}')
ax.hist(pred_ft_ho.reshape(-1), bins=50, density=True, alpha=0.4, color='darkorange', label=f'Fine-tuned  W={w_ft_ho:.4f}')
ax.hist(pred_sc_ho.reshape(-1), bins=50, density=True, alpha=0.4, color='green',      label=f'Scratch     W={w_sc_ho:.4f}')
ax.set_title('A — BG distributions (holdout, normalised)', fontweight='bold')
ax.set_xlabel('Normalised BG'); ax.set_ylabel('Density'); ax.legend(fontsize=8)

# B — 1-day trajectory
ax = axes[0,1]
ns = min(SAMPLES_PER_DAY, len(ho_bg))
ax.plot(ho_bg[:ns,0,0],        color='black',      lw=2.2, label='Real')
ax.plot(pred_pop_ho[:ns,0,0],  color='steelblue',  lw=1.5, label='Population', alpha=0.8)
ax.plot(pred_ft_ho[:ns,0,0],   color='darkorange',  lw=1.5, label='Fine-tuned',  alpha=0.8)
ax.set_title('B — Trajectories (first holdout day)', fontweight='bold')
ax.set_xlabel('Time step (5 min)'); ax.set_ylabel('Normalised BG'); ax.legend()

# C — Training curves
ax = axes[1,0]
ax.plot(ft_history, 's-', color='darkorange', lw=2, label='Fine-tuning')
ax.plot(sc_history, '^-', color='green',      lw=2, label='Scratch')
ax.set_title('C — Training loss curves', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('VAE Loss'); ax.legend()

# D — Wasserstein bar chart
ax = axes[1,1]
names  = ['Population', 'Fine-tuned', 'Scratch']
scores = [w_pop_ho, w_ft_ho, w_sc_ho]
cols   = ['steelblue', 'darkorange', 'green']
bars   = ax.bar(names, scores, color=cols, alpha=0.85, edgecolor='white')
for b, s in zip(bars, scores):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.0002,
            f'{s:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_title('D — Wasserstein distance ↓ (holdout)', fontweight='bold')
ax.set_ylabel('Wasserstein distance')

plt.suptitle(f'{PATIENT_ID} — Population vs Fine-tuned vs Scratch',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🔮 Section 8 — Digital Twin: What-If Simulations

A digital twin is not just a better predictor — it is a **simulation environment**.

Because the cVAE models a *distribution* over trajectories, we can:
1. Sample multiple futures → **uncertainty quantification**
2. Modify therapy inputs and re-simulate → **counterfactual what-if analysis**

The monotonicity constraint ensures all counterfactuals are physiologically plausible.

*Run and explore.*

In [ ]:
def simulate_counterfactual(enc, dec, ins, carb, bg,
                             ins_scale=1.0, carb_scale=1.0, n=20):
    """Generate stochastic trajectories under modified therapy inputs."""
    preds = []
    for _ in range(n):
        zm, zlv = enc([ins, carb, bg], training=False)
        z       = sampling([zm, zlv])
        pred    = dec([z, ins * ins_scale, carb * carb_scale], training=False).numpy()
        preds.append(pred.reshape(-1))
    return np.array(preds)   # (n, N)


# Use first day of holdout
ns = min(SAMPLES_PER_DAY, len(ho_ins))
sim_ins  = ho_ins[:ns]
sim_carb = ho_carb[:ns]
sim_bg   = ho_bg[:ns]

scenarios = {
    'Observed therapy':         dict(ins_scale=1.0,  carb_scale=1.0),
    '+20% insulin dose':        dict(ins_scale=1.2,  carb_scale=1.0),
    '−20% insulin dose':        dict(ins_scale=0.8,  carb_scale=1.0),
    '+20% carbs (larger meals)':dict(ins_scale=1.0,  carb_scale=1.2),
}
colors = ['black', 'steelblue', 'darkorange', 'purple']

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sim_bg[:,0,0], color='red', lw=2.5, label='Real CGM (holdout)', zorder=6)

for (label, kw), col in zip(scenarios.items(), colors):
    preds = simulate_counterfactual(ft_enc, ft_dec, sim_ins, sim_carb, sim_bg, **kw)
    mu    = preds.mean(0)
    sigma = preds.std(0)
    t     = np.arange(len(mu))
    ax.plot(t, mu, color=col, lw=1.8, label=label)
    ax.fill_between(t, mu-sigma, mu+sigma, color=col, alpha=0.12)

ax.set_title(
    f'Digital twin ({PATIENT_ID}) — counterfactual therapy scenarios\n'
    'Shaded bands = ±1 std across 20 stochastic samples (latent uncertainty)',
    fontweight='bold')
ax.set_xlabel('Time step (5 min)'); ax.set_ylabel('Normalised BG')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print('💡 Try editing ins_scale or carb_scale in the scenarios dict above and re-running.')
print('   Monotonicity constraint guarantees the direction is always physiologically correct.')

In [ ]:
# ─── Bonus: How many days of patient data do we actually need? ────────────────
# Replicates Figure 3 of the paper.
# Key finding: most gains happen in the first 1–2 weeks.

max_train_days = P['n_train_days']
days_list = [d for d in [1, 2, 3, 5, 7, 10, 14] if d <= max_train_days]

w_ft_days, w_sc_days = [], []

for nd in days_list:
    ns = nd * SAMPLES_PER_DAY
    ins_sub  = P['tr_ins'][:ns]
    carb_sub = P['tr_carb'][:ns]
    bg_sub   = P['tr_bg'][:ns]

    if ns < 32:
        w_ft_days.append(np.nan); w_sc_days.append(np.nan); continue

    # Fine-tuned
    e1, d1 = build_encoder(), build_decoder()
    d1.set_weights(PRETRAINED_DECODER_WEIGHTS)
    run_training(ins_sub, carb_sub, bg_sub, e1, d1,
                 n_epochs=8, batch_size=32, lr=5e-4, tag=f'{nd}d-ft')
    w1, _ = eval_wasserstein(P['ho_ins'], P['ho_carb'], P['ho_bg'], e1, d1)
    w_ft_days.append(w1)

    # Scratch
    e2, d2 = build_encoder(), build_decoder()
    run_training(ins_sub, carb_sub, bg_sub, e2, d2,
                 n_epochs=8, batch_size=32, lr=5e-4, tag=f'{nd}d-sc')
    w2, _ = eval_wasserstein(P['ho_ins'], P['ho_carb'], P['ho_bg'], e2, d2)
    w_sc_days.append(w2)

    print(f'  {nd:2d} days  fine-tuned={w1:.5f}  scratch={w2:.5f}')

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(days_list, w_ft_days, 'o-', color='darkorange', lw=2, label='Fine-tuned (pretrained prior)')
ax.plot(days_list, w_sc_days, 's-', color='green',      lw=2, label='Scratch (no pretraining)')
w_pop_bonus, _ = eval_population_baseline(P['ho_ins'], P['ho_carb'], P['ho_bg'], ft_dec)
ax.axhline(w_pop_bonus, color='steelblue', ls='--', lw=1.5, label='Population model (holdout)')
ax.set_title('Data efficiency — how many days until personalisation beats the population model?',
             fontweight='bold')
ax.set_xlabel('Training days available for new patient')
ax.set_ylabel('Wasserstein distance (holdout) ↓')
ax.legend(); plt.tight_layout(); plt.show()


---
## 💬 Discussion Questions

1. **Why reinitialise the encoder?**  
   If the decoder carries the population prior, what would happen if we also kept the population encoder weights instead of starting fresh?

2. **Break the monotonicity constraint.**  
   Go back to Blank 3 and remove the negation — set `ins_gain` to `+softplus(ins_raw)` (positive). Re-run Sections 5–8. What happens to the what-if simulation? Why is this a patient safety issue?

3. **VAE vs GAN as population backbone.**  
   The paper compares both. The VAE backbone consistently outperforms the GAN backbone regardless of fine-tuning strategy. Why might a probabilistic encoder produce a *better* transferable prior than an adversarial generator?

4. **Diminishing returns.**  
   In the data efficiency plot, what happens beyond 7–10 days? The paper notes performance can degrade with more data. What two competing effects are at play?

5. **What the latent space absorbs.**  
   This model conditions only on insulin and carbs. Yet it captures patient-specific variability. What physiological factors get implicitly absorbed into `z`? What would it *fundamentally* miss?

---
## 📚 Further reading

- **This workshop's method:** Mujahid et al. — *Personalised Digital Twins for T1D Using Generative AI* (2025)
- **GAN-based T1D simulator:** Mujahid et al. — *Generative deep learning for a T1D simulator* — Commun. Medicine (2024)
- **Monotonicity constraints:** Pellizzari et al. — MetroXRAINE (2025)
- **Exercise-aware digital twin:** Bustos, Mujahid et al. — J. Diabetes Sci. Technol. (2025)
- **ReplayBG (mechanistic baseline):** Cappon et al. — IEEE Trans. Biomed. Eng. (2023)
- **T1DEXI dataset:** [vivli.org](https://vivli.org)